# Drug Response Prediction with Predicted Profiles

Stratified SMILES boxplots (5-fold, predicted-profile training) and Supplementary summary / comparison tables.

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

project_root = Path.cwd()
if not (project_root / "config.py").exists():
    # Running from scripts/04_predict_drug_response/
    project_root = project_root.parents[1]
sys.path.insert(0, str(project_root))
sys.path.insert(0, str(project_root / "scripts" / "04_predict_drug_response"))

from config import config

import importlib
import evaluate_predictions as ev
import create_figures_predictions as figpred

ev = importlib.reload(ev)
figpred = importlib.reload(figpred)

data_dir = str(config.DATA_DIR)
results_dir = str(config.RESULTS_04_DIR)
figures_dir = str(config.FIGURES_04_DIR)
os.makedirs(figures_dir, exist_ok=True)
figpred.figures_dir = figures_dir
figpred.results_dir = results_dir

In [2]:
ALL_FAMILY = [
    "CPA",
    "chemCPA",
    "PRnet",
    "GEARS",
    "scFoundation",
    "Average effect",
    "No effect",
]
SMILES_ONLY_MEASURED = ["Pre+SMILES", "Post+SMILES", "LFC+SMILES"]
SMILES_ONLY_PREDICTED = ["Post+SMILES", "LFC+SMILES"]
FAMILY_LABEL = "ALL family SMILES"


def _load_split_cv_smiles(dataset: str) -> pd.DataFrame:
    path = os.path.join(results_dir, f"{dataset}_split_cv_smiles_predictions.csv")
    df = pd.read_csv(path, index_col=0)
    df["train_set"] = np.where(df["profile_source"] == "observed", "Measured", "Predicted")
    df = df.rename(columns={"profile_source": "test_set"})
    smiles = {
        "pre_treatment_smiles",
        "post_treatment_smiles",
        "LFC_smiles",
        *SMILES_ONLY_MEASURED,
        *SMILES_ONLY_PREDICTED,
    }
    df = df.loc[df["model"].astype(str).isin(smiles)].copy()
    # Predicted-profile training only (SMILES job used --no-include-observed).
    return df.loc[df["train_set"].astype(str).ne("Measured")].copy()


sciplex_split_cv_smiles = _load_split_cv_smiles("sciplex")
mcfarland_split_cv_smiles = _load_split_cv_smiles("mcfarland")
print(
    "SMILES models:",
    sorted(
        set(sciplex_split_cv_smiles["model"].astype(str))
        | set(mcfarland_split_cv_smiles["model"].astype(str))
    ),
)
print(
    "SMILES sources:",
    sorted(
        set(sciplex_split_cv_smiles["test_set"].astype(str))
        | set(mcfarland_split_cv_smiles["test_set"].astype(str))
    ),
)

SMILES models: ['LFC_smiles', 'post_treatment_smiles']
SMILES sources: ['CPA_predicted', 'GEARS_predicted', 'PRnet_predicted', 'average_effect', 'chemCPA_predicted', 'no_effect', 'scFoundation_predicted']


## Stratified Pearson: within-line / within-tissue / within-drug

One figure: SciPlex within-cell-line, McFarland within-tissue, McFarland within-drug.
All boxplots share the same group set. Measured Pre+SMILES from T1 (5-fold); No effect injected when available.

In [ ]:
_prev_measured = figpred.MEASURED_MODEL_ORDER
_prev_predicted = figpred.PREDICTED_MODEL_ORDER
figpred.MEASURED_MODEL_ORDER = SMILES_ONLY_MEASURED
figpred.PREDICTED_MODEL_ORDER = SMILES_ONLY_PREDICTED
try:
    fig, axes = figpred.plot_family_custom_panels(
        "predictions_smiles_stratified_line_tissue_drug",
        ALL_FAMILY,
        [
            {
                "predictions": sciplex_split_cv_smiles,
                "dataset": "sciplex",
                "stratum": "per_cell_line",
                "label": "SciPlex3\n(within cell line)",
                "ylabel": r"Within-line Pearson r  $\rightarrow$",
                "show_ylabel": True,
            },
            {
                "predictions": mcfarland_split_cv_smiles,
                "dataset": "mcfarland",
                "stratum": "per_cell_line",
                "label": "McFarland\n(within tissue)",
                "ylabel": r"Within-tissue Pearson r  $\rightarrow$",
                "show_ylabel": True,
            },
            {
                "predictions": mcfarland_split_cv_smiles,
                "dataset": "mcfarland",
                "stratum": "per_drug",
                "label": "McFarland\n(within drug)",
                "ylabel": r"Within-drug Pearson r  $\rightarrow$",
                "show_ylabel": True,
            },
        ],
        metric="pearson",
        plot_end_to_end=True,
        inject_measured_t1=True,
        inject_no_effect_t1=True,
    )
    plt.show()
finally:
    figpred.MEASURED_MODEL_ORDER = _prev_measured
    figpred.PREDICTED_MODEL_ORDER = _prev_predicted

## Summary tables: Pearson / Spearman / RMSE + paired comparisons

Mean ± s.d. and paired Δ vs Pre+SMILES, Average effect, and No effect.
Writes CSVs under `results/04_predict_drug_response/` and TeX fragments under
`figures/04_predict_drug_response/` (input by `Supplementary_Data.tex` §5–6).

In [5]:
try:
    from IPython.display import display
except ImportError:
    display = print

CORE_METRICS = ("pearson", "spearman", "rmse")
STRATA = ("pooled", "per_drug", "per_cell_line")


def _display_core_tables(summary: pd.DataFrame, comparisons: pd.DataFrame, family_label: str) -> None:
    print("=" * 100)
    print(family_label)
    print("=" * 100)
    for stratum in STRATA:
        sub = summary[summary["stratum"] == stratum].copy()
        if sub.empty:
            continue
        print(f"\n--- {stratum} (mean ± sd + paired Δ columns) ---")
        metric_cols = []
        for metric in CORE_METRICS:
            metric_cols.extend(
                [
                    c
                    for c in (
                        f"{metric}_mean",
                        f"{metric}_sd",
                        f"{metric}_vs_pre_smiles_delta",
                        f"{metric}_vs_pre_smiles_ci_low",
                        f"{metric}_vs_pre_smiles_ci_high",
                        f"{metric}_vs_pre_smiles_sig",
                        f"{metric}_vs_avg_effect_delta",
                        f"{metric}_vs_avg_effect_ci_low",
                        f"{metric}_vs_avg_effect_ci_high",
                        f"{metric}_vs_avg_effect_sig",
                        f"{metric}_vs_no_effect_delta",
                        f"{metric}_vs_no_effect_ci_low",
                        f"{metric}_vs_no_effect_ci_high",
                        f"{metric}_vs_no_effect_sig",
                    )
                    if c in sub.columns
                ]
            )
        display(sub[["dataset", "test_set", "model", *metric_cols]].round(3))

        comp = comparisons[
            (comparisons["stratum"] == stratum) & (comparisons["metric"].isin(CORE_METRICS))
        ].copy()
        if comp.empty:
            continue
        print(f"\n--- {stratum} paired comparisons (long form) ---")
        comp_cols = [
            c
            for c in (
                "dataset",
                "metric",
                "test_set",
                "model",
                "baseline",
                "mean_diff",
                "ci_low",
                "ci_high",
                "t_pvalue",
                "sig",
                "n_folds",
            )
            if c in comp.columns
        ]
        display(
            comp[comp_cols]
            .sort_values(["dataset", "metric", "test_set", "model", "baseline"])
            .round(4)
        )


summary_tables = []
comparison_tables = []
for dataset_label, dataset_key, preds in (
    ("SciPlex3", "sciplex", sciplex_split_cv_smiles),
    ("McFarland", "mcfarland", mcfarland_split_cv_smiles),
):
    preds = figpred.replace_measured_predictions_with_t1(preds, dataset_key)
    out = ev.build_stratified_summary_tables(
        preds,
        dataset_label=dataset_label,
        family=FAMILY_LABEL,
        predicted_categories=ALL_FAMILY,
        dataset=dataset_key,
        data_dir=data_dir,
        include_cpa_end_to_end=True,
        comparison_metrics=CORE_METRICS,
    )
    summary_tables.append(out["summary"])
    if len(out["comparisons"]):
        comparison_tables.append(out["comparisons"])

all_summary = pd.concat(summary_tables, ignore_index=True)
all_comparisons = (
    pd.concat(comparison_tables, ignore_index=True) if comparison_tables else pd.DataFrame()
)

summary_path = os.path.join(results_dir, "predictions_stratified_summary_tables.csv")
comparisons_path = os.path.join(results_dir, "predictions_stratified_paired_comparisons.csv")
all_summary.to_csv(summary_path, index=False)
all_comparisons.to_csv(comparisons_path, index=False)
print(f"Wrote {summary_path}")
print(f"Wrote {comparisons_path}")

tex_paths = figpred.write_summary_tables_tex(
    all_summary,
    all_comparisons,
    outdir=figures_dir,
    family=FAMILY_LABEL,
)
for path in tex_paths:
    print(f"Wrote {path}")

figures_path = Path(figures_dir)
family_masters = [
    p for p in tex_paths if p.name.startswith("predictions_stratified_summary_tables_")
]
combined_master = figures_path / "predictions_stratified_summary_tables.tex"
combined_master.write_text(
    "\n".join(
        [
            "% Auto-generated stratified summary / comparison tables.",
            "% Requires: booktabs, longtable, pdflscape (for wide pages).",
            "",
            *[rf"\input{{{p.name}}}" for p in family_masters],
            "",
        ]
    ),
    encoding="utf-8",
)
print(f"Wrote {combined_master}")

_display_core_tables(all_summary, all_comparisons, FAMILY_LABEL)

2026-09-17 11:03:10,158 - evaluate_predictions - INFO - align_stratified_metric_groups: 45 common groups across 17 panels (excluded test_sets=[]); example panels=[('Average effect', 'LFC+SMILES'), ('Average effect', 'Post+SMILES'), ('CPA', 'LFC+SMILES'), ('CPA', 'Post+SMILES'), ('GEARS', 'LFC+SMILES'), ('GEARS', 'Post+SMILES')]
2026-09-17 11:03:10,748 - evaluate_predictions - INFO - align_stratified_metric_groups: 3 common groups across 17 panels (excluded test_sets=[]); example panels=[('Average effect', 'LFC+SMILES'), ('Average effect', 'Post+SMILES'), ('CPA', 'LFC+SMILES'), ('CPA', 'Post+SMILES'), ('GEARS', 'LFC+SMILES'), ('GEARS', 'Post+SMILES')]
2026-09-17 11:03:11,527 - evaluate_predictions - INFO - align_stratified_metric_groups: 45 common groups across 18 panels (excluded test_sets=[]); example panels=[('Average effect', 'LFC+SMILES'), ('Average effect', 'Post+SMILES'), ('CPA', 'LFC+SMILES'), ('CPA', 'Post+SMILES'), ('CPA\nEnd-to-End', 'Embedding'), ('GEARS', 'LFC+SMILES')]
202

Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\results\04_predict_drug_response\predictions_stratified_summary_tables.csv
Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\results\04_predict_drug_response\predictions_stratified_paired_comparisons.csv


2026-09-17 11:04:06,465 - create_figures_predictions - INFO - Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\figures\04_predict_drug_response\predictions_stratified_summary_all_family_smiles_pooled_rmse.tex (36 rows)
2026-09-17 11:04:06,581 - create_figures_predictions - INFO - Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\figures\04_predict_drug_response\predictions_stratified_comparisons_all_family_smiles_pooled.tex (282 rows)
2026-09-17 11:04:06,658 - create_figures_predictions - INFO - Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\figures\04_predict_drug_response\predictions_stratified_summary_all_family_smiles_per_drug_pearson.tex (36 rows)
2026-09-17 11:04:06,732 - create_figures_predictions - INFO - Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\figures\04_predict_drug_response\predictions_stratified_summary_all_family_smiles_per_drug_spearman.tex (36 rows)
2026-09-17 11:04:06,850 - create_figures_pred

Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\figures\04_predict_drug_response\predictions_stratified_summary_all_family_smiles_pooled_pearson.tex
Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\figures\04_predict_drug_response\predictions_stratified_summary_all_family_smiles_pooled_spearman.tex
Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\figures\04_predict_drug_response\predictions_stratified_summary_all_family_smiles_pooled_rmse.tex
Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\figures\04_predict_drug_response\predictions_stratified_summary_all_family_smiles_pooled.tex
Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\figures\04_predict_drug_response\predictions_stratified_comparisons_all_family_smiles_pooled.tex
Wrote u:\nbrouwer1\perturbation_modeling\drug-response-prediction\figures\04_predict_drug_response\predictions_stratified_summary_all_family_smiles_per_drug_pearson.tex
Wrote u:\n

,dataset,test_set,model,pearson_mean,pearson_sd,pearson_vs_pre_smiles_delta,pearson_vs_pre_smiles_ci_low,pearson_vs_pre_smiles_ci_high,pearson_vs_pre_smiles_sig,pearson_vs_avg_effect_delta,...,rmse_vs_pre_smiles_ci_high,rmse_vs_pre_smiles_sig,rmse_vs_avg_effect_delta,rmse_vs_avg_effect_ci_low,rmse_vs_avg_effect_ci_high,rmse_vs_avg_effect_sig,rmse_vs_no_effect_delta,rmse_vs_no_effect_ci_low,rmse_vs_no_effect_ci_high,rmse_vs_no_effect_sig
0,SciPlex3,Measured,LFC+SMILES,0.687,0.073,0.443,0.204,0.681,**,0.125,...,-0.026,**,-0.048,-0.093,-0.003,*,-0.048,-0.088,-0.008,*
1,SciPlex3,Measured,Post+SMILES,0.502,0.087,0.257,-0.097,0.610,NS,-0.061,...,-0.000,*,-0.020,-0.056,0.015,NS,-0.020,-0.056,0.015,NS
2,SciPlex3,Measured,Pre+SMILES,0.245,0.227,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,SciPlex3,Average effect,LFC+SMILES,0.562,0.105,0.318,-0.064,0.699,NS,NaN,...,0.052,NS,NaN,NaN,NaN,NaN,-0.000,-0.010,0.010,NS
4,SciPlex3,CPA,LFC+SMILES,0.736,0.056,0.491,0.144,0.838,*,0.173,...,0.016,NS,-0.034,-0.042,-0.025,**,-0.034,-0.043,-0.025,**
5,SciPlex3,GEARS,LFC+SMILES,0.673,0.082,0.428,0.083,0.773,*,0.110,...,0.002,NS,-0.021,-0.065,0.022,NS,-0.022,-0.062,0.019,NS
6,SciPlex3,No effect,LFC+SMILES,0.561,0.080,0.316,-0.038,0.670,NS,-0.001,...,0.047,NS,0.000,-0.010,0.010,NS,NaN,NaN,NaN,NaN
7,SciPlex3,PRnet,LFC+SMILES,0.648,0.100,0.403,0.041,0.764,*,0.085,...,0.041,NS,-0.016,-0.032,-0.000,*,-0.016,-0.029,-0.003,*
8,SciPlex3,chemCPA,LFC+SMILES,0.826,0.018,0.581,0.294,0.868,**,0.263,...,-0.015,*,-0.062,-0.076,-0.047,**,-0.062,-0.082,-0.042,**
9,SciPlex3,scFoundation,LFC+SMILES,0.645,0.110,0.400,0.013,0.787,*,0.082,...,0.026,NS,-0.016,-0.046,0.014,NS,-0.017,-0.042,0.009,NS


...

,dataset,metric,test_set,model,baseline,mean_diff,ci_low,ci_high,t_pvalue,sig,n_folds
571,McFarland,pearson,Average effect,LFC+SMILES,No effect,0.2727,0.1052,0.4403,0.0051,**,10
570,McFarland,pearson,Average effect,LFC+SMILES,Pre+SMILES,0.0479,-0.0618,0.1577,0.3489,NS,10
590,McFarland,pearson,Average effect,Post+SMILES,No effect,-0.0283,-0.0592,0.0025,0.0676,NS,10
589,McFarland,pearson,Average effect,Post+SMILES,Pre+SMILES,-0.0283,-0.0592,0.0025,0.0675,NS,10
573,McFarland,pearson,CPA,LFC+SMILES,Average effect,0.2133,0.0030,0.4237,0.0475,*,10
...,...,...,...,...,...,...,...,...,...,...,...
212,SciPlex3,spearman,scFoundation,LFC+SMILES,No effect,0.1548,-0.0659,0.3755,0.1646,NS,45
210,SciPlex3,spearman,scFoundation,LFC+SMILES,Pre+SMILES,-0.0274,-0.2050,0.1503,0.7576,NS,45
230,SciPlex3,spearman,scFoundation,Post+SMILES,Average effect,-0.0556,-0.2164,0.1053,0.4900,NS,45
231,SciPlex3,spearman,scFoundation,Post+SMILES,No effect,-0.0556,-0.2164,0.1053,0.4900,NS,45



--- per_cell_line (mean ± sd + paired Δ columns) ---


,dataset,test_set,model,pearson_mean,pearson_sd,pearson_vs_pre_smiles_delta,pearson_vs_pre_smiles_ci_low,pearson_vs_pre_smiles_ci_high,pearson_vs_pre_smiles_sig,pearson_vs_avg_effect_delta,...,rmse_vs_pre_smiles_ci_high,rmse_vs_pre_smiles_sig,rmse_vs_avg_effect_delta,rmse_vs_avg_effect_ci_low,rmse_vs_avg_effect_ci_high,rmse_vs_avg_effect_sig,rmse_vs_no_effect_delta,rmse_vs_no_effect_ci_low,rmse_vs_no_effect_ci_high,rmse_vs_no_effect_sig
36,SciPlex3,Measured,LFC+SMILES,0.416,0.438,0.177,-0.669,1.022,NS,-0.127,...,0.082,NS,-0.040,-0.136,0.056,NS,-0.040,-0.146,0.066,NS
37,SciPlex3,Measured,Post+SMILES,0.363,0.187,0.123,-0.056,0.302,NS,-0.180,...,-0.000,*,-0.020,-0.066,0.026,NS,-0.020,-0.066,0.026,NS
38,SciPlex3,Measured,Pre+SMILES,0.239,0.117,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
39,SciPlex3,Average effect,LFC+SMILES,0.543,0.014,0.304,0.047,0.560,*,NaN,...,0.065,NS,NaN,NaN,NaN,NaN,0.000,-0.009,0.010,NS
40,SciPlex3,CPA,LFC+SMILES,0.708,0.060,0.468,0.118,0.818,*,0.165,...,0.056,NS,-0.032,-0.086,0.023,NS,-0.031,-0.095,0.033,NS
41,SciPlex3,GEARS,LFC+SMILES,0.636,0.019,0.397,0.129,0.664,*,0.093,...,0.046,NS,-0.018,-0.081,0.045,NS,-0.017,-0.087,0.053,NS
42,SciPlex3,No effect,LFC+SMILES,0.586,0.006,0.346,0.065,0.627,*,0.042,...,0.064,NS,-0.000,-0.010,0.009,NS,NaN,NaN,NaN,NaN
43,SciPlex3,PRnet,LFC+SMILES,0.622,0.058,0.383,0.019,0.747,*,0.079,...,0.059,NS,-0.014,-0.050,0.022,NS,-0.013,-0.059,0.032,NS
44,SciPlex3,chemCPA,LFC+SMILES,0.816,0.023,0.576,0.261,0.892,*,0.273,...,0.028,NS,-0.060,-0.115,-0.004,*,-0.060,-0.125,0.006,NS
45,SciPlex3,scFoundation,LFC+SMILES,0.637,0.103,0.397,0.305,0.490,**,0.094,...,0.092,NS,-0.012,-0.069,0.045,NS,-0.011,-0.078,0.055,NS



--- per_cell_line paired comparisons (long form) ---


,dataset,metric,test_set,model,baseline,mean_diff,ci_low,ci_high,t_pvalue,sig,n_folds
712,McFarland,pearson,Average effect,LFC+SMILES,No effect,-0.0159,-0.0419,0.0102,0.2050,NS,11
711,McFarland,pearson,Average effect,LFC+SMILES,Pre+SMILES,-0.0096,-0.0342,0.0150,0.4057,NS,11
731,McFarland,pearson,Average effect,Post+SMILES,No effect,-0.0160,-0.0426,0.0106,0.2098,NS,11
730,McFarland,pearson,Average effect,Post+SMILES,Pre+SMILES,-0.0160,-0.0426,0.0106,0.2099,NS,11
714,McFarland,pearson,CPA,LFC+SMILES,Average effect,0.0355,0.0059,0.0652,0.0236,*,11
...,...,...,...,...,...,...,...,...,...,...,...
353,SciPlex3,spearman,scFoundation,LFC+SMILES,No effect,0.0350,-0.1538,0.2239,0.5084,NS,3
351,SciPlex3,spearman,scFoundation,LFC+SMILES,Pre+SMILES,0.5194,0.4347,0.6042,0.0014,**,3
371,SciPlex3,spearman,scFoundation,Post+SMILES,Average effect,0.0551,-0.1822,0.2924,0.4231,NS,3
372,SciPlex3,spearman,scFoundation,Post+SMILES,No effect,0.0551,-0.1816,0.2918,0.4224,NS,3
